Example of using GeoJSON data sources

# Ecosystems Map

In [ ]:
import logging
import warnings

from lonboard import Map

from rle_python_gee.ecosystems import Ecosystems
from rle_python_gee.aoo import make_aoo_grid
from rle_python_gee.aoo import slugify_ecosystem_name

In [ ]:
warnings.filterwarnings("ignore", module="lonboard._geoarrow.ops.reproject")

logging.getLogger('rle_python_gee.aoo').setLevel(logging.INFO)
logging.basicConfig(level=logging.INFO)

In [ ]:
TEST_EXPORTS = False

## From Shapefile

In [ ]:
# ecosystems = Ecosystems.from_file(
#     '/Users/tylere/Documents/GitHub/RLE-Assessment/rle-python-gee/tests/test_data/colombia_ecosystems.shp',
#     ecosystem_column='ECO_CODE'
# )
# ecosystems.load()
# ecosystems

# if TEST_EXPORTS:
#     ecosystems.to_parquet('/tmp/ecosystems.parquet')

## From parquet file

In [ ]:
import geopandas as gpd
from pathlib import Path
from rle_python_gee import make_ecosystems

remote_url = "https://storage.googleapis.com/rle_test_data/colombia_ecosystems.parquet"
local_path = Path("/tmp/ecosystems.parquet")

if not local_path.exists():
    logging.info(f"Downloading {remote_url} to {local_path}")
    gdf = gpd.read_parquet(remote_url)
    gdf.to_parquet(local_path)
else:
    logging.info(f"Loading {local_path}")
ecosystems = make_ecosystems(local_path, ecosystem_column='ECO_CODE', ecosystem_name_column='ECO_NAME')
ecosystems

In [ ]:
ecosystems.head()

In [ ]:
ecosystems.size()

In [ ]:
ecosystems.ecosystem_names()

In [ ]:
ecosystems.to_map()

In [ ]:
ecosystem_code = 'F1.1.3'
print(ecosystems.ecosystem_name(ecosystem_code))

In [ ]:
# Filter ecosystems then display
ecosystem_column = slugify_ecosystem_name(ecosystem_code)
filtered = ecosystems.filter(ecosystem_code)
print(f'{filtered.size()=}')
filtered.limit(1000).to_map()

In [ ]:
if TEST_EXPORTS:    
    import ee
    ee.Initialize(project='goog-rle-assessments')
    task_id = ecosystems.to_ee_feature_collection(
        'projects/goog-rle-assessments/assets/temp/ecosystems',
        gcs_bucket='rle_test_bucket_1'
    )
    print(f'{task_id=}')

# Create a Ecosystem COG

In [ ]:
ecosystems.to_raster

# EOO Calculations

The Extent of Occurrence (EOO) calculations are described in Section 6.3.2 of the IUCN Guidelines for the application of IUCN Red List of Ecosystems Categories and Criteria (2024).

In [ ]:
from rle_python_gee.eoo import make_eoo

eoo = make_eoo(filtered).compute()

In [ ]:
display(Map(layers=eoo.to_layer() + filtered.to_layer(max_features=4000)))
print(f'{eoo.area_km2 =}')

In [ ]:
filtered.eoo

# AOO Calculations

The Area of Occupancy (AOO) calculations are described in Section 6.3.2 of the IUCN Guidelines for the application of IUCN Red List of Ecosystems Categories and Criteria (2024).

In [ ]:
from pathlib import Path
import geopandas as gpd

USE_AOO_CACHE = True
AOO_CACHE_PATH = Path("/tmp/aoo_grid_cache.parquet")

aoo_grid = make_aoo_grid(ecosystems)

if USE_AOO_CACHE and AOO_CACHE_PATH.exists():
    logging.info(f"Loading AOO grid from cache: {AOO_CACHE_PATH}")
    aoo_grid._grid_cells = gpd.read_parquet(AOO_CACHE_PATH)
    aoo_grid._computed = True
else:
    logging.info("Computing AOO grid (this may take a few minutes)...")
    aoo_grid.compute()
    aoo_grid.to_parquet(AOO_CACHE_PATH)
    logging.info(f"AOO grid cached to {AOO_CACHE_PATH}")

aoo_grid

In [ ]:
aoo_grid.to_map()

In [ ]:
aoo_grid_filtered = aoo_grid.filter_by_ecosystem(ecosystem_code)
aoo_grid_filtered.to_map()

In [ ]:
aoo_grid_filtered.grid_cells.head()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from lonboard.colormap import apply_continuous_cmap

cmap = LinearSegmentedColormap.from_list("white_red", ["white", "red"])
values = aoo_grid_filtered.grid_cells[ecosystem_column].values
normalized = (values - values.min()) / (values.max() - values.min())
colors = apply_continuous_cmap(normalized, cmap)

aoo_grid_filtered.to_map(get_fill_color=colors)

> - Arrange grid cells in ascending order based on their area (smaller first).
> - Calculate accumulated sum of area per cell (‘cumulative area’).

In [ ]:
keep = ['geometry', 'grid_col', 'grid_row', 'count_geoms', 'count_ecosystems', ecosystem_column]
gdf = aoo_grid_filtered.grid_cells[keep].sort_values(by=ecosystem_column)
gdf["cumulative_fraction"] = gdf[ecosystem_column].cumsum()
total_fraction = gdf["cumulative_fraction"].iloc[-1]
gdf["cumulative_proportion"] = gdf["cumulative_fraction"] / total_fraction
gdf

> - Calculate AOO by counting the number of cells with a ‘cumulative proportion’ greater than 0.01 (i.e. exclude cells that in combination account for up to 1% of the total mapped extent of the ecosystem type).

In [ ]:
len(gdf)

In [ ]:
aoo_notebook = len(gdf[gdf["cumulative_proportion"] > 0.01])
aoo_notebook

In [ ]:
aoo_direct = filtered.calculate_aoo()
aoo_direct

In [ ]:
assert aoo_notebook == aoo_direct

## AOO Grid Exports

In [ ]:
if TEST_EXPORTS:
    aoo_grid.to_parquet('/tmp/aoo_grid.parquet')

In [ ]:
if TEST_EXPORTS:
    task_id = aoo_grid.to_ee_feature_collection(
        'projects/goog-rle-assessments/assets/temp/aoo_grid',
        gcs_bucket='rle_test_bucket_1'
    )
    print(f'{task_id=}')

# AOO Grid Polygons

In [ ]:
AOO_POLYGONS_CACHE_PATH = Path("/tmp/aoo_grid_polygons_cache.parquet")

aoo_grid_polygons = aoo_grid.to_polygons()

if USE_AOO_CACHE and AOO_POLYGONS_CACHE_PATH.exists():
    logging.info(f"Loading AOO grid polygons from cache: {AOO_POLYGONS_CACHE_PATH}")
    aoo_grid_polygons._polygons = gpd.read_parquet(AOO_POLYGONS_CACHE_PATH)
    aoo_grid_polygons._computed = True
else:
    logging.info("Computing AOO grid polygons (this may take a few minutes)...")
    aoo_grid_polygons.compute()
    aoo_grid_polygons.to_parquet(AOO_POLYGONS_CACHE_PATH)
    logging.info(f"AOO grid polygons cached to {AOO_POLYGONS_CACHE_PATH}")

aoo_grid_polygons

In [ ]:
aoo_grid_polygons.to_map()

In [ ]:
aoo_grid_polygons.filter_by_ecosystem(ecosystem_code).to_map()

In [ ]:
if TEST_EXPORTS:
    # Write the grid polygons to a parquet file
    aoo_grid_polygons.to_parquet('/tmp/aoo_grid_polygons.parquet')

In [ ]:
if TEST_EXPORTS:
    task_id = aoo_grid_polygons.to_ee_feature_collection(
        'projects/goog-rle-assessments/assets/temp/aoo_grid_polygons',
        gcs_bucket='rle_test_bucket_1'
    )
    print(f'{task_id=}')